In [13]:
# ============================================================
# Hybrid Rocket Fuel — Model Comparison
# Same data, same split → XGBoost vs Random Forest vs LightGBM
# Metrics: R2, MAE, RMSE per target on the TEST set
# Exports: results/model_comparison.csv + model_comparison.txt
# MC Dropout is NOT included here (uncertainty only; separate scripts)
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [14]:
# ============================================================
# CONFIG  (keep in sync with your single-model scripts)
# ============================================================

EXCEL_PATH = Path("daq_raw_4blends.xlsx")
OUTPUT_DIR = Path("results")
OUTPUT_CSV = OUTPUT_DIR / "model_comparison.csv"
OUTPUT_TXT = OUTPUT_DIR / "model_comparison.txt"

SHEETS_TO_USE = None
REQUIRED_SHEET_COUNT = 4
FUEL_COLUMN = "fuel_type"

FEATURE_COLUMNS = ["X_Value", "Force", "Pressure_0", "Gox"]

TARGET_COLUMNS = ["Pressure", "rdot_mm_s", "Isp", "Thrust", "Cf", "cstar", "Specific Density", "O/F Ratio"]

SEED = 100
TEST_SIZE = 0.20
MIN_LABELED_ROWS = 20

# Shared-ish capacity for a fair-ish comparison
N_ESTIMATORS = 300

np.random.seed(SEED)


In [15]:
# ============================================================
# HELPERS
# ============================================================

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def clean_numeric(df, columns):
    out = df.copy()
    for c in columns:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
        else:
            out[c] = np.nan
    return out


def validate_excel_file(path: Path):
    if not path.exists():
        raise FileNotFoundError(
            f"Excel not found: {path.resolve()}\n"
            "Put daq_raw_4blends.xlsx next to this script."
        )
    size = path.stat().st_size
    if size < 100:
        raise ValueError(f"File too small ({size} bytes): {path}")
    with open(path, "rb") as f:
        magic = f.read(4)
    if magic[:2] != b"PK":
        raise ValueError(
            f"Not a valid .xlsx file: {path.resolve()}\n"
            "Save As → Excel Workbook (.xlsx) and re-upload."
        )


def load_dataset():
    validate_excel_file(EXCEL_PATH)
    xls = pd.ExcelFile(EXCEL_PATH, engine="openpyxl")
    sheet_names = xls.sheet_names if SHEETS_TO_USE is None else list(SHEETS_TO_USE)

    if len(sheet_names) < REQUIRED_SHEET_COUNT:
        raise ValueError(
            f"Need >= {REQUIRED_SHEET_COUNT} sheets; found {len(sheet_names)}. "
            f"Present: {xls.sheet_names}"
        )
    sheet_names = sheet_names[:REQUIRED_SHEET_COUNT]

    frames = []
    for sheet in sheet_names:
        raw = pd.read_excel(xls, sheet_name=sheet, engine="openpyxl")
        missing = [c for c in FEATURE_COLUMNS if c not in raw.columns]
        if missing:
            raise ValueError(
                f"Sheet '{sheet}' missing inputs {missing}. "
                f"Columns: {list(raw.columns)}"
            )
        keep = list(FEATURE_COLUMNS)
        for t in TARGET_COLUMNS:
            if t not in raw.columns:
                raw[t] = np.nan
            if t not in keep:
                keep.append(t)
        keep = list(dict.fromkeys(keep))
        frame = clean_numeric(raw[keep], keep)
        frame[FUEL_COLUMN] = str(sheet)
        frames.append(frame)

    data = pd.concat(frames, ignore_index=True)
    data = data.dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)
    if "X_Value" in data.columns:
        data = data[data["X_Value"] >= 0].reset_index(drop=True)

    labeled_mask = data[TARGET_COLUMNS].notna().all(axis=1)
    for t in TARGET_COLUMNS:
        labeled_mask &= np.isfinite(pd.to_numeric(data[t], errors="coerce"))

    n_labeled = int(labeled_mask.sum())
    print(f"Sheets used      : {sheet_names}")
    print(f"Total input rows : {len(data)}")
    print(f"Labeled rows     : {n_labeled}")
    if n_labeled < MIN_LABELED_ROWS:
        raise ValueError(
            f"Need >= {MIN_LABELED_ROWS} labeled rows; found {n_labeled}."
        )
    return data, sheet_names, labeled_mask


def build_pipeline(model):
    preprocessor = ColumnTransformer(
        transformers=[
            ("fuel", make_one_hot_encoder(), [FUEL_COLUMN]),
            ("numeric", "passthrough", FEATURE_COLUMNS),
        ],
        remainder="drop",
    )
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", MultiOutputRegressor(model)),
    ])


def make_models(seed):
    return {
        "XGBoost": XGBRegressor(
            n_estimators=N_ESTIMATORS,
            max_depth=3,
            learning_rate=0.05,
            min_child_weight=5,
            reg_alpha=1.0,
            reg_lambda=5.0,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=seed,
            n_jobs=-1,
            verbosity=0,
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            max_depth=12,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=seed,
            n_jobs=-1,
        ),
        "LightGBM": LGBMRegressor(
            n_estimators=N_ESTIMATORS,
            max_depth=3,
            learning_rate=0.05,
            num_leaves=8,
            min_child_samples=10,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=5.0,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
        ),
    }


def safe_r2(y_true, y_pred):
    if np.unique(y_true).size < 2:
        return np.nan
    return float(r2_score(y_true, y_pred))


def evaluate_model(name, model, x_train, y_train, x_test, y_test, target_names):
    pipe = build_pipeline(model)
    pipe.fit(x_train, y_train)
    pred = pipe.predict(x_test)

    rows = []
    for i, t in enumerate(target_names):
        yt = y_test[:, i]
        yp = pred[:, i]
        rows.append({
            "model": name,
            "target": t,
            "r2": safe_r2(yt, yp),
            "mae": float(mean_absolute_error(yt, yp)),
            "rmse": float(np.sqrt(mean_squared_error(yt, yp))),
        })
    return pd.DataFrame(rows), pipe


In [16]:
# ============================================================
# MAIN
# ============================================================

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    data, sheet_names, labeled_mask = load_dataset()
    labeled = data.loc[labeled_mask].reset_index(drop=True)

    x_lab = labeled[FEATURE_COLUMNS + [FUEL_COLUMN]].copy()
    y_lab = labeled[TARGET_COLUMNS].to_numpy(dtype=float)

    counts = labeled[FUEL_COLUMN].value_counts()
    stratify = labeled[FUEL_COLUMN] if counts.min() >= 2 else None

    x_train, x_test, y_train, y_test = train_test_split(x_lab, y_lab, test_size=TEST_SIZE, random_state=SEED, stratify=stratify)
    print(f"Train: {len(x_train)} | Test: {len(x_test)}")
    print(f"Targets: {TARGET_COLUMNS}")
    print(f"Models : XGBoost, RandomForest, LightGBM")
    print("-" * 60)

    all_metrics = []
    for name, base_model in make_models(SEED).items():
        print(f"Training {name}...")
        metrics_df, _ = evaluate_model(name, base_model, x_train, y_train, x_test, y_test, TARGET_COLUMNS)
        all_metrics.append(metrics_df)
        for _, row in metrics_df.iterrows():
            r2 = row["r2"]
            r2_s = f"{r2:.4f}" if np.isfinite(r2) else "nan"
            print(
                f"  {row['target']:<12}  R2={r2_s:>8}  "
                f"MAE={row['mae']:.4f}  RMSE={row['rmse']:.4f}"
            )
        print()

    comparison = pd.concat(all_metrics, ignore_index=True)
    comparison = comparison.sort_values(["target", "model"]).reset_index(drop=True)
    comparison.to_csv(OUTPUT_CSV, index=False)

    # Wide pivot for README / poster (R2 focus)
    pivot_r2 = comparison.pivot(index="target", columns="model", values="r2")
    pivot_mae = comparison.pivot(index="target", columns="model", values="mae")
    pivot_rmse = comparison.pivot(index="target", columns="model", values="rmse")

    print("=" * 60)
    print("TEST R2 (higher is better)")
    print(pivot_r2.to_string(float_format=lambda v: f"{v: .4f}"))
    print()
    print("TEST MAE (lower is better)")
    print(pivot_mae.to_string(float_format=lambda v: f"{v: .4f}"))
    print()
    print("TEST RMSE (lower is better)")
    print(pivot_rmse.to_string(float_format=lambda v: f"{v: .4f}"))

    # Winner per target by R2 (then lower RMSE as tie-break)
    winners = []
    for t in TARGET_COLUMNS:
        sub = comparison[comparison["target"] == t].copy()
        sub = sub.sort_values(["r2", "rmse"], ascending=[False, True])
        best = sub.iloc[0]
        winners.append({
            "target": t,
            "best_model": best["model"],
            "best_r2": best["r2"],
            "best_mae": best["mae"],
            "best_rmse": best["rmse"],
        })
    winners_df = pd.DataFrame(winners)

    print()
    print("BEST MODEL PER TARGET (by test R2)")
    print(winners_df.to_string(index=False))

    lines = [
        "HYBRID ROCKET FUEL — MODEL COMPARISON",
        "=" * 60,
        f"Excel        : {EXCEL_PATH}",
        f"Sheets       : {sheet_names}",
        f"Features     : {FEATURE_COLUMNS}",
        f"Targets      : {TARGET_COLUMNS}",
        f"Train / Test : {len(x_train)} / {len(x_test)}",
        f"Seed         : {SEED}",
        f"n_estimators : {N_ESTIMATORS}",
        "",
        "TEST R2",
        "-" * 40,
        pivot_r2.to_string(float_format=lambda v: f"{v: .4f}"),
        "",
        "TEST MAE",
        "-" * 40,
        pivot_mae.to_string(float_format=lambda v: f"{v: .4f}"),
        "",
        "TEST RMSE",
        "-" * 40,
        pivot_rmse.to_string(float_format=lambda v: f"{v: .4f}"),
        "",
        "BEST MODEL PER TARGET",
        "-" * 40,
        winners_df.to_string(index=False),
        "",
        f"CSV saved: {OUTPUT_CSV.resolve()}",
        "",
    ]
    OUTPUT_TXT.write_text("\n".join(lines), encoding="utf-8")

    print()
    print(f"Saved: {OUTPUT_CSV.resolve()}")
    print(f"Saved: {OUTPUT_TXT.resolve()}")


if __name__ == "__main__":
    main()


Sheets used      : ['Firing 01', 'Firing 02', 'Firing 03', 'Firing 04']
Total input rows : 524
Labeled rows     : 524
Train: 419 | Test: 105
Targets: ['Pressure', 'rdot_mm_s', 'Isp', 'Thrust', 'Cf', 'cstar', 'Specific Density', 'O/F Ratio']
Models : XGBoost, RandomForest, LightGBM
------------------------------------------------------------
Training XGBoost...
  Pressure      R2= -0.0124  MAE=0.0223  RMSE=0.0351
  rdot_mm_s     R2=  0.9983  MAE=0.0137  RMSE=0.0265
  Isp           R2=  0.9841  MAE=2.9566  RMSE=6.2042
  Thrust        R2=  0.9935  MAE=0.1201  RMSE=0.2770
  Cf            R2= -0.0003  MAE=0.0001  RMSE=0.0003
  cstar         R2=  0.6862  MAE=107.2405  RMSE=161.1149
  Specific Density  R2=  0.9851  MAE=2648.2143  RMSE=5356.8835
  O/F Ratio     R2=  0.9605  MAE=0.1991  RMSE=0.3206

Training RandomForest...
  Pressure      R2=  0.1243  MAE=0.0200  RMSE=0.0327
  rdot_mm_s     R2=  0.9934  MAE=0.0374  RMSE=0.0527
  Isp           R2=  0.9277  MAE=6.6913  RMSE=13.2397
  Thrust     